In [ ]:
# Machine-specific paths live in config.yaml next to this notebook (gitignored).
# Copy config.example.yaml -> config.yaml and fill in your paths.
from phoreal.config import load_config
cfg = load_config('config.yaml')

### Import packages

In [ ]:
#Import general libs you'll need
import os
import geopandas as gpd
from shapely.geometry import Point

# Import functions from main
from phoreal.io import readers
from phoreal import analysis
from phoreal.agg_configs import ATL_AGG_CONFIG

# Import functions from the utils
from phoreal import plotting as plotter

### Identify files



In [ ]:
atl03_dir = cfg['florida_atl03_dir']
atl03_name = 'ATL03_20200212094736_07280607_006_01.h5'
atl03_file = os.path.join(atl03_dir, atl03_name)

# Define ATL08 File
atl08_dir = cfg['florida_atl08_dir']
atl08_name = 'ATL08_20200212094736_07280607_006_01.h5'
atl08_file = os.path.join(atl08_dir, atl08_name)

# Define ATL24 file
atl24_dir = cfg['florida_atl24_dir']
atl24_name = 'ATL24_20200212094736_07280607_006_01_001_01.h5'
atl24_file = os.path.join(atl24_dir, atl24_name)

# Define the ground track of interest
gt = 'gt1r'

### Get photon rate dataframe


In [ ]:
# Get photon rate DF
df_ph = readers.read_photon_dataframe(atl03_file, gt, atl08_file, atl24_file)

### Plot output 

Well, it works on my machine

In [ ]:
import matplotlib.pyplot as plt
plotter.plot_alongtrack(df_ph.alongtrack, df_ph.h_ph, df_ph.combined_class,'Example Plot')

### Plot Contested Classes

In [ ]:
plotter.plot_alongtrack_contested_classes(df_ph.alongtrack,df_ph.h_ph,df_ph.contested_class,'Contested Classes')

In [ ]:
plotter.plot_alongtrack_atl08_atl24_contested(df_ph.alongtrack,df_ph.h_ph,df_ph.atl08_class,df_ph.atl24_class,df_ph.contested_class,'ATL08/24 and Contested Classes')

### Aggregate data

This function will aggregate the data into 10 m segments

In [ ]:
# Aggregate photon rate DF to 10 m segment 
df_seg = analysis.aggregate_by_segment(df_ph, ATL_AGG_CONFIG, res=10)

### Write out the data

Write out the data as a geopandas dataframe if needed

In [ ]:
# Write out as geopandas dataframe
geometry_seg = [Point(xy) for xy in zip(df_seg.longitude, df_seg.latitude)]
gdf_seg = gpd.GeoDataFrame(df_seg, geometry=geometry_seg, crs="EPSG:4326")

# Save geopandas dataframe
gdf_seg.to_file("is2_topobathy.gpkg", layer='atl08atl24', driver="GPKG")